In [1]:
with open(r'Solar_Industries_India_Comprehensive_Analysis.txt','r',encoding='utf-8') as f:
    text = f.read()


In [2]:
print("Lenght of Data :- " , len(text))

Lenght of Data :-  138115


In [3]:
print(text[:1000])

# Solar Industries India Limited: A Comprehensive Analysis of India's Leading Explosives and Defense Manufacturer

 
*Date: November 8, 2025*

## Table of Contents

1. [Executive Summary](#executive-summary)
2. [Company Overview and Historical Background](#company-overview-and-historical-background)
3. [Business Segments and Product Portfolio](#business-segments-and-product-portfolio)
4. [Financial Performance Analysis](#financial-performance-analysis)
5. [Market Position and Competitive Landscape](#market-position-and-competitive-landscape)
6. [Management and Corporate Governance](#management-and-corporate-governance)
7. [Global Presence and International Expansion](#global-presence-and-international-expansion)
8. [Recent Developments and Strategic Achievements](#recent-developments-and-strategic-achievements)
9. [Industry Analysis and Market Dynamics](#industry-analysis-and-market-dynamics)
10. [Future Outlook and Growth Prospects](#future-outlook-and-growth-prospects)
11. [Investmen

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 "#$%&'()*+,-./0123456789:<>ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyzô–₹据票
88


In [5]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("My name is Aryan"))
print(decode(encode("My name is Aryan")))


[41, 81, 1, 70, 57, 69, 61, 1, 65, 75, 1, 29, 74, 81, 57, 70]
My name is Aryan


In [ ]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape , data.dtype)
print(data[:1000])

In [ ]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
block_size = 8  # COntext window
train_data[:block_size+1]

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'when input is {context} the target is  {target}')

In [10]:
torch.manual_seed(1473)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) 
    return x , y

xb , yb = get_batch("train")
print("inputs:")
print(xb.shape)
print(xb)
print("target:")
print(yb.shape)
print(yb)

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b , :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target is: {target}")

inputs:
torch.Size([4, 8])
tensor([[65, 71, 70, 28, 17, 19, 27, 15],
        [75, 61,  1, 76, 61, 59, 64, 70],
        [75, 76, 74, 57,  1, 59, 71, 77],
        [61,  1, 81, 61, 57, 74, 75, 12]])
target:
torch.Size([4, 8])
tensor([[71, 70, 28, 17, 19, 27, 15, 59],
        [61,  1, 76, 61, 59, 64, 70, 71],
        [76, 74, 57,  1, 59, 71, 77, 70],
        [ 1, 81, 61, 57, 74, 75, 12,  1]])
when input is [65] the target is: 71
when input is [65, 71] the target is: 70
when input is [65, 71, 70] the target is: 28
when input is [65, 71, 70, 28] the target is: 17
when input is [65, 71, 70, 28, 17] the target is: 19
when input is [65, 71, 70, 28, 17, 19] the target is: 27
when input is [65, 71, 70, 28, 17, 19, 27] the target is: 15
when input is [65, 71, 70, 28, 17, 19, 27, 15] the target is: 59
when input is [75] the target is: 61
when input is [75, 61] the target is: 1
when input is [75, 61, 1] the target is: 76
when input is [75, 61, 1, 76] the target is: 61
when input is [75, 61, 1, 76, 6

In [11]:
print(xb) #our input to transformer

tensor([[65, 71, 70, 28, 17, 19, 27, 15],
        [75, 61,  1, 76, 61, 59, 64, 70],
        [75, 76, 74, 57,  1, 59, 71, 77],
        [61,  1, 81, 61, 57, 74, 75, 12]])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1473)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx)  # (B, T, C)

        B, T, C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)

        loss = F.cross_entropy(logits, targets)

        return logits, loss


m = BigramLanguageModel(vocab_size)
logits , loss = m(xb,yb)
print(logits.shape)
print(loss)